# 🚐 CareMax Capstone Project — Trip Timing & Duration Analysis
### ⏱ Histogram + Pareto Chart for Trip Duration (Pickup → Dropoff)

**Purpose:** This notebook calculates the **duration of patient trips in minutes** by subtracting pickup time from dropoff time. The differences are grouped into fixed 10-minute intervals (0–210 min), flagged for 60+ minute early pickups, cumulative Pareto statistics are generated, and the distribution is visualized using ggplot2 in R.


## 1️⃣ Load Required R Libraries
Loads necessary visualization helpers for histogram bars and percentage axes.

In [ ]:
library(ggplot2)
library(scales)

## 2️⃣ Optional Sample Data
Illustrative example showing how subtraction handles NA values. This is not used in the fleet analysis plot but can help test logic.

In [ ]:
data <- data.frame(
  column1 = c(5, NA, 8, 2),
  column2 = c(2, 4, NA, 1)
)
head(data)

## 3️⃣ Load Transportation Logs
`all_data1.csv` should be placed in the working directory. Expected columns (in seconds): `Dropoff.Perform` and `Pickup.Perform`.

In [ ]:
all_data1 <- read.csv("all_data1.csv")
head(all_data1)

## 4️⃣ Compute Trip Duration (Minutes)
Subtracts pickup timestamp from dropoff timestamp and converts to minutes. Flags if trip was >60 minutes early.

In [ ]:
subtracted_minutes <- (all_data1$Dropoff.Perform - all_data1$Pickup.Perform) / 60

pareto_data <- data.frame(
  Minutes = subtracted_minutes,
  Greater_than_60 = ifelse(subtracted_minutes > 60, "Yes", "No")
)

pareto_data$Minutes <- as.numeric(pareto_data$Minutes)
pareto_data <- pareto_data[!is.na(pareto_data$Minutes), ]
head(pareto_data)

## 5️⃣ Define 10-Minute Interval Bins (0–210 Minutes)

In [ ]:
interval_breaks <- seq(0, 210, by = 10)

pareto_data$Interval <- cut(
  pareto_data$Minutes,
  breaks = interval_breaks,
  labels = paste(interval_breaks[-length(interval_breaks)], "-", interval_breaks[-1], sep="")
)

head(pareto_data$Interval)

## 6️⃣ Count Pickup Timing Violations per Interval
Produces frequency table: **Interval × (Trip >60 min early?)**

In [ ]:
counts <- table(pareto_data$Interval, pareto_data$Greater_than_60)
pareto_counts <- as.data.frame(counts)
head(pareto_counts)

## 7️⃣ Compute Cumulative Pareto Stats

In [ ]:
pareto_counts$cumulative <- ave(pareto_counts$Freq, pareto_counts$Var1, FUN = cumsum)
head(pareto_counts)

## 8️⃣ Create Pareto Visualization — Trip Duration Histogram + Cumulative Trend

In [ ]:
pareto_plot <- ggplot(pareto_counts, aes(x = Var1, y = Freq, fill = Var2)) +
  geom_bar(stat = "identity") +
  geom_text(aes(label = Freq), vjust = -0.5, size = 3) +
  geom_line(aes(x = Var1, y = cumulative, group = Var2, color = Var2), linetype = "dashed") +
  geom_point(aes(x = Var1, y = cumulative, group = Var2, color = Var2), size = 2) +
  scale_y_continuous(sec.axis = sec_axis(~./sum(pareto_counts$Freq), labels = percent_format())) +
  labs(
    x = "Time Interval (minutes)",
    y = "Frequency",
    fill = "Greater than 60 minutes early?",
    color = "Greater than 60 minutes early?",
    title = "⏱ Trip Duration by 10-Minute Interval — CareMax Fleet"
  ) +
  theme_minimal() +
  theme(legend.position = "bottom")

print(pareto_plot)